# import

In [4]:
from sklearn import linear_model
import sys
from sklearn import linear_model
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import pyreadr 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f
from scipy.stats import chi2
import math
from scipy.stats import gmean
import re


In [24]:
# Step 2: DATA COLLECTION

df = pd.read_json("./data_raw/netherlands.json")
df_petrol = pd.read_json("data_raw/NL_petrol_ 1565_items.json")
df_diesel = pd.read_json("data_raw/NL_diesel_ 262_items.json")
df_electric = pd.read_json("data_raw/NL_electric_200_items.json")
df_hybrid = pd.read_json("data_raw/NL_hybrid_201_items.json")
df_plugin = pd.read_json("data_raw/NL_plugin_200_items.json")

df = pd.concat([df_petrol, df_diesel, df_electric, df_hybrid, df_plugin], ignore_index=True)

### required attributes: price Y, km Y , year_of_prod Y, age Y, fuel_type, vehicle type
df['country'] = 'NL'
df['price'] = df['price'].apply(lambda x: x['total']['amount'])
df['km'] = df['attributes'].apply(lambda x: int(re.sub(r'\D', '', x.get('Mileage'))))

df['first_reg'] = df['attributes'].apply(
    lambda x: round(
        int(x.get('First Registration').split('/')[1]) +
        (int(x.get('First Registration').split('/')[0]) - 0.5) / 12,
        2
    )
)

df['age'] = round(2026 - df['first_reg'],1)

df['fuel_raw'] = df['attributes'].apply(
    lambda x: x.get('Fuel') if x.get('Fuel') else 'Full electric'
)

df['transmission'] = df['attributes'].apply(lambda x: x.get('Transmission'))

df


,title,previewImage,segment,category,brand,model,url,price,createdDate,modifiedDate,...,id,sellerId,priceRating,dealerDetails,country,km,first_reg,age,fuel_raw,transmission
0,Mercedes-Benz CLA 250 Shooting Brake Aut 4-Mat...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,EstateCar,Mercedes-Benz,CLA 250 Shooting Brake,https://suchen.mobile.de/fahrzeuge/details.htm...,20250.00,2024-11-28T16:01:11.000Z,2026-03-17T18:16:38.000Z,...,410101225,454577,"{'rating': 'Increased price', 'priceRanges': {...","{'id': 454577, 'name': 'AUTO WIENTJES B.V.', '...",NL,99953,2016.46,9.5,Petrol,Automatic
1,Mercedes-Benz E 200 AMG Line | 95.600km NAP | ...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,Limousine,Mercedes-Benz,E 200,https://suchen.mobile.de/fahrzeuge/details.htm...,23900.00,2025-11-27T12:03:05.000Z,2026-03-17T17:01:58.000Z,...,442474247,11129564,"{'rating': 'Fair price', 'priceRanges': {'Very...","{'id': 11129564, 'name': 'SELDENRIJK B.V.', 's...",NL,95614,2016.46,9.5,Petrol,Automatic
2,Audi Q3 1.4 TFSI CoD S Edition S-Line | NAP | ...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,Audi,Q3,https://suchen.mobile.de/fahrzeuge/details.htm...,17995.00,2026-01-05T12:11:04.000Z,2026-03-17T17:05:11.000Z,...,445072975,12302052,"{'rating': 'Increased price', 'priceRanges': {...","{'id': 12302052, 'name': 'AutoJorg', 'sellerTy...",NL,117206,2015.46,10.5,Petrol,Automatic
3,BMW X3 xDrive35i Executive 306 PK Memory / Hea...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,BMW,X3,https://suchen.mobile.de/fahrzeuge/details.htm...,21950.00,2026-01-20T09:41:04.000Z,2026-03-17T17:19:22.000Z,...,446151134,22321638,"{'rating': 'Increased price', 'priceRanges': {...","{'id': 22321638, 'name': 'Auto Mooij BV', 'sel...",NL,122495,2014.21,11.8,Petrol,Automatic
4,BMW M240i 2 Serie Cabrio | M-Sportpakket | xen...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,Cabrio,BMW,M240i,https://suchen.mobile.de/fahrzeuge/details.htm...,27900.00,2025-09-01T13:39:12.000Z,2026-03-17T17:19:18.000Z,...,435271396,31656123,"{'rating': 'Very good price', 'priceRanges': {...","{'id': 31656123, 'name': 'Ekris Retail B.V.', ...",NL,86000,2017.38,8.6,Petrol,Automatic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2424,Volvo XC40 1.5 T4 Recharge R Design Expression...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,Volvo,XC40,https://suchen.mobile.de/fahrzeuge/details.htm...,20699.47,2026-01-22T14:42:05.000Z,2026-03-17T16:16:18.000Z,...,446352961,11129564,"{'rating': 'Increased price', 'priceRanges': {...","{'id': 11129564, 'name': 'SELDENRIJK B.V.', 's...",NL,176441,2021.04,5.0,"Hybrid (petrol/electric), Plug-in hybrid",Automatic
2425,Volvo XC60 2.0 Recharge T6 AWD Momentum | pano...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,Volvo,XC60,https://suchen.mobile.de/fahrzeuge/details.htm...,28500.34,2025-11-11T14:36:05.000Z,2026-03-17T17:18:14.000Z,...,441187132,11129564,"{'rating': 'Fair price', 'priceRanges': {'Very...","{'id': 11129564, 'name': 'SELDENRIJK B.V.', 's...",NL,156879,2021.88,4.1,"Hybrid (petrol/electric), Plug-in hybrid",Automatic
2426,Lynk&Co 01 1.5 (Plug-In) (INCL-BTW) Aut. *PANO...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,Lynk&Co,01,https://suchen.mobile.de/fahrzeuge/details.htm...,17345.35,2026-01-07T07:27:05.000Z,2026-03-17T15:39:30.000Z,...,445190554,454618,"{'rating': 'Very good price', 'priceRanges': {...","{'id': 454618, 'name': 'HAVERKAMPS AUTOMOBIELE...",NL,160937,2022.62,3.4,"Hybrid (petrol/electric), Plug-in hybrid",Automatic
2427,Renault Megane Estate 1.6 E-Tech Plug-In Hybri...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,EstateCar,Renault,Megane,https://suchen.mobile.de/fahrzeuge/details.htm...,19749.62,2026-01-28T15:09:05.000Z,2026-03-17T16:10:43.000Z,...,446844833,13848366,"{'rating': 'Fair price', 'priceRanges': {'Very...","{'id': 13848366, 'name': 'AUTOBEDRIJF VAN SCHA...",NL,57071,2023.04,3.0,"Hybri

In [26]:
# Step 3: DATA PREPARATION




# filter out: old, very low mileage cars (over 10 years, under 50.000 km)
df = df[~((df['age'] > 10) & (df['km'] < 50000))]

def clean_fuel(fuel):
    if not fuel:
        return 'Electric'  # default if missing
    
    fuel = fuel.lower()
    
    if 'electric' in fuel and 'hybrid' not in fuel:
        return 'Electric'
    elif 'plug-in' in fuel:
        return 'Plug-in Hybrid'
    elif 'hybrid' in fuel:
        return 'Hybrid'
    elif 'petrol' in fuel:
        return 'Petrol'
    elif 'diesel' in fuel:
        return 'Diesel'
    else:
        return 'Other'

df['fuel'] = df['fuel_raw'].apply(clean_fuel)

df

,title,previewImage,segment,category,brand,model,url,price,createdDate,modifiedDate,...,sellerId,priceRating,dealerDetails,country,km,first_reg,age,fuel_raw,transmission,fuel
0,Mercedes-Benz CLA 250 Shooting Brake Aut 4-Mat...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,EstateCar,Mercedes-Benz,CLA 250 Shooting Brake,https://suchen.mobile.de/fahrzeuge/details.htm...,20250.00,2024-11-28T16:01:11.000Z,2026-03-17T18:16:38.000Z,...,454577,"{'rating': 'Increased price', 'priceRanges': {...","{'id': 454577, 'name': 'AUTO WIENTJES B.V.', '...",NL,99953,2016.46,9.5,Petrol,Automatic,Petrol
1,Mercedes-Benz E 200 AMG Line | 95.600km NAP | ...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,Limousine,Mercedes-Benz,E 200,https://suchen.mobile.de/fahrzeuge/details.htm...,23900.00,2025-11-27T12:03:05.000Z,2026-03-17T17:01:58.000Z,...,11129564,"{'rating': 'Fair price', 'priceRanges': {'Very...","{'id': 11129564, 'name': 'SELDENRIJK B.V.', 's...",NL,95614,2016.46,9.5,Petrol,Automatic,Petrol
2,Audi Q3 1.4 TFSI CoD S Edition S-Line | NAP | ...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,Audi,Q3,https://suchen.mobile.de/fahrzeuge/details.htm...,17995.00,2026-01-05T12:11:04.000Z,2026-03-17T17:05:11.000Z,...,12302052,"{'rating': 'Increased price', 'priceRanges': {...","{'id': 12302052, 'name': 'AutoJorg', 'sellerTy...",NL,117206,2015.46,10.5,Petrol,Automatic,Petrol
3,BMW X3 xDrive35i Executive 306 PK Memory / Hea...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,BMW,X3,https://suchen.mobile.de/fahrzeuge/details.htm...,21950.00,2026-01-20T09:41:04.000Z,2026-03-17T17:19:22.000Z,...,22321638,"{'rating': 'Increased price', 'priceRanges': {...","{'id': 22321638, 'name': 'Auto Mooij BV', 'sel...",NL,122495,2014.21,11.8,Petrol,Automatic,Petrol
4,BMW M240i 2 Serie Cabrio | M-Sportpakket | xen...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,Cabrio,BMW,M240i,https://suchen.mobile.de/fahrzeuge/details.htm...,27900.00,2025-09-01T13:39:12.000Z,2026-03-17T17:19:18.000Z,...,31656123,"{'rating': 'Very good price', 'priceRanges': {...","{'id': 31656123, 'name': 'Ekris Retail B.V.', ...",NL,86000,2017.38,8.6,Petrol,Automatic,Petrol
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2424,Volvo XC40 1.5 T4 Recharge R Design Expression...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,Volvo,XC40,https://suchen.mobile.de/fahrzeuge/details.htm...,20699.47,2026-01-22T14:42:05.000Z,2026-03-17T16:16:18.000Z,...,11129564,"{'rating': 'Increased price', 'priceRanges': {...","{'id': 11129564, 'name': 'SELDENRIJK B.V.', 's...",NL,176441,2021.04,5.0,"Hybrid (petrol/electric), Plug-in hybrid",Automatic,Plug-in Hybrid
2425,Volvo XC60 2.0 Recharge T6 AWD Momentum | pano...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,Volvo,XC60,https://suchen.mobile.de/fahrzeuge/details.htm...,28500.34,2025-11-11T14:36:05.000Z,2026-03-17T17:18:14.000Z,...,11129564,"{'rating': 'Fair price', 'priceRanges': {'Very...","{'id': 11129564, 'name': 'SELDENRIJK B.V.', 's...",NL,156879,2021.88,4.1,"Hybrid (petrol/electric), Plug-in hybrid",Automatic,Plug-in Hybrid
2426,Lynk&Co 01 1.5 (Plug-In) (INCL-BTW) Aut. *PANO...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,OffRoad,Lynk&Co,01,https://suchen.mobile.de/fahrzeuge/details.htm...,17345.35,2026-01-07T07:27:05.000Z,2026-03-17T15:39:30.000Z,...,454618,"{'rating': 'Very good price', 'priceRanges': {...","{'id': 454618, 'name': 'HAVERKAMPS AUTOMOBIELE...",NL,160937,2022.62,3.4,"Hybrid (petrol/electric), Plug-in hybrid",Automatic,Plug-in Hybrid
2427,Renault Megane Estate 1.6 E-Tech Plug-In Hybri...,https://img.classistatic.de/api/v1/mo-prod/ima...,Car,EstateCar,Renault,Megane,https://suchen.mobile.de/fahrzeuge/details.htm...,19749.62,2026-01-28T15:09:05.000Z,2026-03-17T16:10:43.000Z,...,13848366,"{'rating': 'Fair price', 'priceRanges': {'Very...","{'id': 13848366, 'name': 'AUTOBEDRIJF VAN SCHA...",NL,57071,2023.04,3.0,"Hybrid (petro

In [27]:
df.to_json('NL_prepared_data')

In [29]:
# data overview

print(df['fuel'].value_counts())


fuel
Petrol            1557
Plug-in Hybrid     344
Diesel             262
Electric           201
Hybrid              57
Name: count, dtype: int64
